# Phase 10-11: Generation & Evaluation

Inference dan evaluasi memakai sitasi natural dari schema chunk baru, tanpa ID `[R#]` di jawaban.


## Mapping Fase ke Implementasi Saat Ini

Pipeline produksi di `pipeline_legal_rag_indonesia.md` mendefinisikan Phase 1-11. Implementasi notebook ini memakai varian praktis ChromaDB:

- Phase 1-3: dikerjakan oleh `praproses_ringan_pasal.py`.
- Phase 4: dikerjakan oleh `finalize_chunks_for_chroma.py`, menghasilkan `../data/processed_chunks_ringan_pasal_chroma_ready.json`.
- Phase 5: embedding memakai `embedding_text` dengan `intfloat/multilingual-e5-base`.
- Phase 6: vector store memakai ChromaDB lokal sesuai proposal final.
- Phase 7: hybrid retrieval memakai dense Chroma + BM25 lokal.
- Phase 8: reranker dibuat opsional. Default mati supaya notebook ringan.
- Phase 9: context assembler memakai `display_text` + `citation_text`; sibling expansion bisa ditambah setelah chunk ayat/sibling tersedia stabil.
- Phase 10: generation wajib menyebut sumber hukum secara natural, misalnya `Peraturan Pemerintah No. 35 Tahun 2021, Pasal 56`.
- Phase 11: evaluasi inference + hook RAGAS.


In [ ]:
import hashlib
import json
import os
import pickle
import re
from pathlib import Path
from typing import Dict, List, Any

import chromadb
import numpy as np
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

DATA_PATH = Path("../data/processed_chunks_ringan_pasal_chroma_ready.json")
CHROMA_DB_DIR = Path("../data/chroma_db")
COLLECTION_NAME = "hukum_ketenagakerjaan"
BM25_PATH = Path("../data/bm25_index.pkl")
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"Data: {DATA_PATH}")
print(f"Chroma: {CHROMA_DB_DIR} / {COLLECTION_NAME}")

In [ ]:
def load_chunks(path: Path = DATA_PATH) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"{path} tidak ditemukan. Jalankan praproses_ringan_pasal.py lalu finalize_chunks_for_chroma.py dulu.")
    chunks = json.loads(path.read_text(encoding="utf-8"))
    required = {"id", "text", "display_text", "embedding_text", "citation_text", "metadata"}
    missing = [i for i, c in enumerate(chunks[:20]) if not required.issubset(c)]
    if missing:
        raise ValueError(f"Chunk belum pakai schema baru. Cek index sample: {missing}")
    return ensure_unique_chunk_ids(chunks)


def ensure_unique_chunk_ids(chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = {}
    fixed = 0
    for idx, chunk in enumerate(chunks):
        base_id = str(chunk.get("id") or f"chunk-{idx}")
        count = seen.get(base_id, 0)
        seen[base_id] = count + 1
        if count:
            seed = "::".join([
                base_id,
                str(idx),
                chunk.get("metadata", {}).get("source_file", ""),
                chunk.get("metadata", {}).get("pasal_id", ""),
                chunk.get("text", "")[:200],
            ])
            chunk["original_id"] = base_id
            chunk["id"] = hashlib.sha1(seed.encode("utf-8")).hexdigest()
            fixed += 1
    if fixed:
        print(f"Fixed duplicated chunk ids in memory: {fixed}")
    return chunks


def normalize_metadata(chunk: Dict[str, Any]) -> Dict[str, Any]:
    meta = dict(chunk.get("metadata", {}))
    meta["chunk_id"] = chunk.get("id", "")
    meta["citation_text"] = chunk.get("citation_text", "")
    meta["source_file"] = meta.get("source_file", "")
    meta["pasal_id"] = meta.get("pasal_id", "")
    meta["bab"] = meta.get("bab", "")
    meta["bab_title"] = meta.get("bab_title", "")
    meta["regulation_type"] = meta.get("regulation_type", "")
    meta["nomor"] = meta.get("nomor", "")
    meta["tentang"] = meta.get("tentang", "")
    meta["year"] = int(meta.get("year") or 0)
    meta["publication_year"] = int(meta.get("publication_year") or meta.get("year") or 0)
    safe = {}
    for key, value in meta.items():
        if value is None:
            safe[key] = ""
        elif isinstance(value, (str, int, float, bool)):
            safe[key] = value
        else:
            safe[key] = json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else str(value)
    return safe


def tokenize_for_bm25(text: str) -> List[str]:
    return re.findall(r"[a-zA-Z0-9]+", text.lower())


def compact_citation(meta: Dict[str, Any]) -> str:
    reg_type = str(meta.get("regulation_type") or "Aturan").strip()
    nomor = str(meta.get("nomor") or "").strip()
    year = str(meta.get("publication_year") or meta.get("year") or "").strip()
    pasal = str(meta.get("pasal_id") or "").strip()

    parts = [reg_type]
    if nomor and nomor.lower() != "unknown":
        parts.append(f"No. {nomor}")
    if year and year != "0":
        parts.append(f"Tahun {year}")
    citation = " ".join(parts).strip()
    if pasal:
        citation = f"{citation}, {pasal}"
    return citation


def build_reference(meta: Dict[str, Any], idx: int | None = None) -> str:
    prefix = f"[R{idx}] " if idx is not None else ""
    return prefix + compact_citation(meta)

chunks = load_chunks()
print(f"Loaded {len(chunks)} chunks from {DATA_PATH}")
print("Sample citation:", chunks[0]["citation_text"])

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
client = chromadb.PersistentClient(path=str(CHROMA_DB_DIR))
collection = client.get_collection(COLLECTION_NAME)

bm25 = None
bm25_ids = []
if BM25_PATH.exists():
    with BM25_PATH.open("rb") as f:
        payload = pickle.load(f)
    bm25 = payload["bm25"]
    bm25_ids = payload["ids"]
    print(f"BM25 loaded: {len(bm25_ids)} docs")
else:
    print("BM25 index tidak ditemukan. Retrieval tetap jalan dengan dense Chroma saja.")


def dense_search(query: str, fetch_k: int = 30) -> List[Dict[str, Any]]:
    q_emb = embedding_model.encode(
        ["query: " + query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0].tolist()
    res = collection.query(
        query_embeddings=[q_emb],
        n_results=fetch_k,
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for doc_id, doc, meta, dist in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"id": doc_id, "text": doc, "metadata": meta, "dense_distance": float(dist), "source": "dense"})
    return hits


def bm25_search(query: str, fetch_k: int = 30) -> List[Dict[str, Any]]:
    if bm25 is None:
        return []
    scores = bm25.get_scores(tokenize_for_bm25(query))
    order = np.argsort(scores)[::-1][:fetch_k]
    ids = [bm25_ids[i] for i in order if scores[i] > 0]
    if not ids:
        return []
    got = collection.get(ids=ids, include=["documents", "metadatas"])
    lookup = {doc_id: (doc, meta) for doc_id, doc, meta in zip(got["ids"], got["documents"], got["metadatas"])}
    hits = []
    for i in order:
        doc_id = bm25_ids[i]
        if scores[i] <= 0 or doc_id not in lookup:
            continue
        doc, meta = lookup[doc_id]
        hits.append({"id": doc_id, "text": doc, "metadata": meta, "bm25_score": float(scores[i]), "source": "bm25"})
    return hits


def rrf_fuse(result_sets: List[List[Dict[str, Any]]], weights: List[float] | None = None, rrf_k: int = 60) -> List[Dict[str, Any]]:
    weights = weights or [1.0] * len(result_sets)
    fused = {}
    for hits, weight in zip(result_sets, weights):
        for rank, hit in enumerate(hits, 1):
            item = fused.setdefault(hit["id"], {"score": 0.0, "hit": hit})
            item["score"] += weight / (rrf_k + rank)
    out = []
    for item in sorted(fused.values(), key=lambda x: x["score"], reverse=True):
        hit = item["hit"]
        hit["rrf_score"] = item["score"]
        out.append(hit)
    return out


def lex_posterior_score(hit: Dict[str, Any]) -> float:
    meta = hit.get("metadata", {})
    score = float(hit.get("rrf_score", 0.0))

    try:
        year = int(meta.get("publication_year") or meta.get("year") or 0)
    except Exception:
        year = 0
    try:
        hierarchy = int(meta.get("regulation_hierarchy") or 99)
    except Exception:
        hierarchy = 99

    if str(meta.get("active_status", "")).lower() == "berlaku":
        score += 0.030
    score += min(max(year - 2000, 0), 40) * 0.001
    score += max(0, 6 - hierarchy) * 0.003

    # Jangan buang OCR/noisy docs karena bisa saja dokumen penting seperti PP 35/2021.
    # Cukup penalti ringan supaya dokumen bersih naik jika relevansinya mirip.
    if meta.get("quality_status") == "needs_review":
        score -= 0.010

    return score


def dedupe_legal_hits(hits: List[Dict[str, Any]], k: int = 8) -> List[Dict[str, Any]]:
    ranked = sorted(hits, key=lex_posterior_score, reverse=True)
    seen = set()
    out = []
    for hit in ranked:
        meta = hit.get("metadata", {})
        key = (
            meta.get("source_file", ""),
            meta.get("pasal_id", ""),
            meta.get("chunk_kind", ""),
            meta.get("chunk_index", ""),
        )
        if key in seen:
            continue
        seen.add(key)
        hit["final_score"] = lex_posterior_score(hit)
        out.append(hit)
        if len(out) >= k:
            break
    return out

def retrieve_documents(query: str, k: int = 6, fetch_k: int = 30, use_bm25: bool = True) -> List[Dict[str, Any]]:
    dense_hits = dense_search(query, fetch_k=fetch_k)
    sparse_hits = bm25_search(query, fetch_k=fetch_k) if use_bm25 else []
    fused = rrf_fuse([dense_hits, sparse_hits], weights=[1.0, 0.7]) if sparse_hits else dense_hits
    return dedupe_legal_hits(fused, k=k)


def print_references(docs: List[Dict[str, Any]]) -> None:
    seen = set()
    for doc in docs:
        citation = compact_citation(doc["metadata"])
        if citation in seen:
            continue
        seen.add(citation)
        print(f"- {citation}")

# Smoke test
smoke_docs = retrieve_documents("berapa pesangon pekerja yang di PHK", k=5)
print_references(smoke_docs)

In [ ]:
# Phase 8 - Optional reranker.
# Default mati supaya notebook tetap ringan. Aktifkan kalau ingin download model reranker.
USE_RERANKER = False
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
reranker = None

if USE_RERANKER:
    from sentence_transformers import CrossEncoder
    reranker = CrossEncoder(RERANKER_MODEL_NAME, device=DEVICE)
    print("Reranker aktif:", RERANKER_MODEL_NAME)
else:
    print("Reranker neural nonaktif. Retrieval memakai RRF dense+BM25 + dedupe legal.")


def rerank_documents(query: str, docs: List[Dict[str, Any]], k: int = 6) -> List[Dict[str, Any]]:
    if reranker is None or not docs:
        return docs[:k]
    pairs = [(query, d["text"]) for d in docs]
    scores = reranker.predict(pairs)
    for doc, score in zip(docs, scores):
        doc["rerank_score"] = float(score)
    return sorted(docs, key=lambda x: x.get("rerank_score", 0.0), reverse=True)[:k]


In [ ]:
# Phase 9 - Context assembler.
# Assembler memakai label sumber hukum internal, tetapi jawaban tetap memakai sitasi natural.
# Kalau chunk split panjang berasal dari pasal yang sama, dedupe_legal_hits sudah menjaga urutan dan chunk_index.
def assemble_context(docs: List[Dict[str, Any]], max_docs: int = 6) -> List[Dict[str, Any]]:
    assembled = []
    seen = set()
    for doc in docs:
        meta = doc.get("metadata", {})
        key = (meta.get("source_file", ""), meta.get("pasal_id", ""), meta.get("chunk_index", ""), doc.get("id", ""))
        if key in seen:
            continue
        seen.add(key)
        assembled.append(doc)
        if len(assembled) >= max_docs:
            break
    return assembled


def retrieve_context(query: str, k: int = 6, fetch_k: int = 30) -> List[Dict[str, Any]]:
    candidates = retrieve_documents(query, k=max(k * 2, 10), fetch_k=fetch_k)
    reranked = rerank_documents(query, candidates, k=max(k * 2, 10))
    return assemble_context(reranked, max_docs=k)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LLM_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # ganti ke model lokal/lebih besar kalau VRAM cukup
USE_4BIT = torch.cuda.is_available()

quantization_config = None
if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID, trust_remote_code=True)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    quantization_config=quantization_config,
    trust_remote_code=True,
)
print("LLM ready:", LLM_MODEL_ID)

In [ ]:
def sanitize_output(text: str) -> str:
    text = re.sub(r"```.*?```", "", text, flags=re.DOTALL)
    text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)
    text = re.sub(r"\*(.*?)\*", r"\1", text)
    text = re.sub(r"^#{1,6}\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"\s*\[(?:R\d+(?:\s*,\s*)?)+\]", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\b(?:Menurut|Berdasarkan)\s+(?:referensi|konteks)\b[:,]?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\n\s*[-=*]{3,}\s*\n", "\n", text)
    text = re.sub(r"(?:\n\s*Referensi\s*:\s*)[\s\S]*$", "", text, flags=re.IGNORECASE)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def build_context(docs: List[Dict[str, Any]]) -> str:
    parts = []
    for i, doc in enumerate(docs, 1):
        meta = doc["metadata"]
        citation = compact_citation(meta)
        clean_text = str(doc["text"]).strip()
        parts.append(f"SUMBER HUKUM {i}: {citation}\n{clean_text}")
    return "\n\n".join(parts)


def _normalize_reg_type(value: str) -> str:
    value = str(value or "").lower()
    if "undang" in value or value == "uu":
        return "uu"
    if "pemerintah" in value or value == "pp":
        return "pp"
    if "presiden" in value or "perpres" in value:
        return "perpres"
    if "menteri" in value or "permen" in value:
        return "permen"
    return value


def validate_named_citations(answer: str, docs: List[Dict[str, Any]]) -> Dict[str, Any]:
    pattern = re.compile(
        r"(Undang-Undang|Peraturan Pemerintah|Peraturan Presiden|Peraturan Menteri(?:\s+Ketenagakerjaan)?|UU|PP|Perpres|Permen)"
        r"\s+(?:No\.?|Nomor)?\s*([A-Za-z0-9./-]+)\s+Tahun\s+(\d{4})",
        re.IGNORECASE,
    )
    cited = []
    invalid = []
    for reg_type, nomor, year in pattern.findall(answer):
        label = f"{reg_type} No. {nomor} Tahun {year}"
        cited.append(label)
        target_type = _normalize_reg_type(reg_type)
        found = False
        for doc in docs:
            meta = doc.get("metadata", {})
            doc_type = _normalize_reg_type(meta.get("regulation_type", ""))
            doc_nomor = str(meta.get("nomor", "")).strip()
            doc_year = str(meta.get("publication_year") or meta.get("year") or "").strip()
            if doc_type == target_type and doc_nomor == str(nomor).strip() and doc_year == str(year):
                found = True
                break
        if not found:
            invalid.append(label)
    return {"cited": sorted(set(cited)), "invalid": sorted(set(invalid)), "ok": not invalid}


def build_prompt(query: str, docs: List[Dict[str, Any]]) -> str:
    context = build_context(docs)
    return f"""Anda adalah Pakar Hukum Ketenagakerjaan Indonesia.
ATURAN WAJIB (GROUNDING KETAT):
1. Jawab HANYA berdasarkan KONTEKS. Jika jawaban tidak ada di konteks, katakan referensi yang tersedia belum cukup.
2. Dilarang mengarang nomor pasal, nama aturan, hak, rumus pesangon, atau kesimpulan yang tidak muncul di konteks.
3. Sebutkan sumber hukum secara natural di dalam kalimat, contoh: Peraturan Pemerintah No. 35 Tahun 2021, Pasal 56.
4. DILARANG memakai kode sitasi [R1], [R2], dan sejenisnya di jawaban.
5. DILARANG menulis daftar referensi di dalam jawaban. Daftar referensi akan dicetak sistem secara terpisah.
6. Jangan menulis frasa seperti "Menurut referensi" atau "Berdasarkan konteks". Langsung jawab substansinya.
7. Jika ada beberapa aturan relevan, utamakan aturan yang lebih baru dan lebih spesifik, lalu jelaskan ringkas alasannya.
8. Gunakan bahasa Indonesia yang jelas, teks polos, dan jangan mengulang sitasi.

KONTEKS:
{context}

PERTANYAAN:
{query}

JAWABAN:"""


def generate_answer(query: str, k: int = 6, max_new_tokens: int = 650) -> Dict[str, Any]:
    docs = retrieve_context(query, k=k)
    prompt = build_prompt(query, docs)
    inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=12000).to(llm_model.device)
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.12,
        no_repeat_ngram_size=6,
    )
    raw = llm_tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    answer = sanitize_output(raw)
    validation = validate_named_citations(answer, docs)
    return {"query": query, "answer": answer, "references": docs, "validation": validation}

result = generate_answer("Jika pekerja di-PHK karena pelanggaran berat, berapa pesangonnya?", k=6)
print(result["answer"])
print("\nReferensi:")
print_references(result["references"])
print("\nValidation:", result["validation"])


In [ ]:
eval_questions = [
    "Jika pekerja di-PHK karena pelanggaran berat, berapa pesangonnya?",
    "Apa kewajiban pengusaha terkait alat pelindung diri?",
    "Apa prinsip pelaksanaan SOP administrasi pemerintahan?",
    "Bagaimana aturan keselamatan dan kesehatan kerja lingkungan kerja?",
]

rows = []
for q in eval_questions:
    out = generate_answer(q, k=6, max_new_tokens=700)
    rows.append({
        "question": q,
        "answer": out["answer"],
        "contexts": [d["text"] for d in out["references"]],
        "reference_citations": [build_reference(d["metadata"], i) for i, d in enumerate(out["references"], 1)],
        "citation_ids_used": out["validation"]["cited"],
        "invalid_citation_ids": out["validation"]["invalid"],
    })

import pandas as pd
eval_df = pd.DataFrame(rows)
eval_df

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_relevancy, faithfulness, context_precision, context_recall

# Isi ground_truth kalau mau context_recall bermakna. Kalau kosong, pakai metrik tanpa context_recall.
ragas_rows = []
for row in rows:
    ragas_rows.append({
        "question": row["question"],
        "answer": row["answer"],
        "contexts": row["contexts"],
        "ground_truth": "",
    })

dataset = Dataset.from_list(ragas_rows)
metrics = [faithfulness, answer_relevancy, context_precision]
# Tambahkan context_recall hanya jika ground_truth diisi manual.
# metrics.append(context_recall)

# Butuh konfigurasi LLM/embedding RAGAS sesuai environment. Jalankan setelah env RAGAS siap.
# score = evaluate(dataset, metrics=metrics)
# score.to_pandas()

## Perbandingan dengan backup `03_Phase_10_11_Generation_Eval.ipynb`

Yang dipertahankan dari notebook lama:
- grounding ketat ke konteks;
- hybrid retrieval dense + BM25;
- citation validation;
- opsi model 4-bit untuk GPU.

Yang diubah:
- sumber data lama diganti ke `../data/processed_chunks_ringan_pasal_chroma_ready.json`;
- teks embedding memakai `embedding_text`, bukan `text` mentah;
- dokumen yang masuk prompt memakai `display_text`;
- referensi memakai `citation_text` ringkas dan jawaban menyebut sumber hukum secara natural, sehingga lebih gampang dibaca;
- filter lama `section_type == batang_tubuh` dihapus karena schema chunk baru tidak punya field itu. Kualitas dikontrol saat finalisasi melalui quarantine dan split pasal.
